# E0: Frozen-Split RGB-Only YOLOv8s Baseline

This notebook is the executable E0 experiment only. It uses the frozen recording-level split and the validated frame-level RGB pair manifest. It never creates a new split and keeps the two final test contexts out of development cross-validation, training, model selection, and early stopping.

The ZIP is read without extracting unrelated members. Only validated RGB image/annotation pairs referenced by `rgb_dataset_manifest.csv` are materialized into the local YOLO workspace.

In [ ]:
from __future__ import annotations

import csv
import os
import shutil
import sys
import zipfile
from collections import Counter
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SPLIT_MANIFEST = PROJECT_ROOT / "results" / "dataset_split" / "recording_split_manifest.csv"
PAIR_MANIFEST = PROJECT_ROOT / "results" / "rgb_baseline" / "rgb_dataset_manifest.csv"
ZIP_PATH = Path("/content/drive/MyDrive/WiSARD/WiSARDv1.zip")
E0_ROOT = PROJECT_ROOT / "results" / "rgb_baseline" / "e0_yolo_workspace"
RUN_TRAINING = True
EXPECTED_PAIRS = 10_703
EXPECTED_DEVELOPMENT_PAIRS = 9_096
EXPECTED_TEST_PAIRS = 1_607
TEST_CONTEXTS = {"210327_Airfield_FLIR", "210812_Hannegan_Enterprise"}
DEVELOPMENT_CONTEXTS = {
    "210417_MtErie_Enterprise", "210529_Carnation_Enterprise",
    "210924_FHL_Enterprise", "220109_Baker_Enterprise",
}

if not SPLIT_MANIFEST.is_file():
    raise FileNotFoundError(SPLIT_MANIFEST)
if not PAIR_MANIFEST.is_file():
    raise FileNotFoundError(PAIR_MANIFEST)
if not ZIP_PATH.is_file():
    raise FileNotFoundError(ZIP_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"ZIP: {ZIP_PATH}")


In [ ]:
FROZEN_COLUMNS = ["recording_name", "recording_path", "collection_context", "modality", "image_count", "partition", "cv_fold"]
PAIR_COLUMNS = ["image_member_path", "annotation_member_path", "recording_name", "recording_path", "collection_context", "partition", "modality"]

with SPLIT_MANIFEST.open(newline="", encoding="utf-8") as handle:
    frozen_rows = list(csv.DictReader(handle))
with PAIR_MANIFEST.open(newline="", encoding="utf-8") as handle:
    pair_rows = list(csv.DictReader(handle))

if set(frozen_rows[0]) != set(FROZEN_COLUMNS) or set(pair_rows[0]) != set(PAIR_COLUMNS):
    raise ValueError("Manifest columns do not match the frozen E0 contracts.")
if len(pair_rows) != EXPECTED_PAIRS:
    raise ValueError(f"Expected {EXPECTED_PAIRS} RGB pairs, found {len(pair_rows)}")
if Counter(row["partition"] for row in pair_rows) != Counter({"development": EXPECTED_DEVELOPMENT_PAIRS, "test": EXPECTED_TEST_PAIRS}):
    raise ValueError("RGB pair partition counts do not match the frozen E0 counts.")
if any(row["modality"] != "VIS" for row in pair_rows):
    raise ValueError("The RGB pair manifest contains a non-VIS row.")

frozen_by_recording = {row["recording_name"]: row for row in frozen_rows}
for row in pair_rows:
    frozen = frozen_by_recording.get(row["recording_name"])
    if frozen is None:
        raise ValueError(f"Pair recording missing from frozen recording split: {row['recording_name']}")
    for field in ("recording_path", "collection_context", "partition", "modality"):
        if row[field] != frozen[field]:
            raise ValueError(f"Frozen assignment mismatch for {row['recording_name']} field {field}")

pair_contexts = {row["collection_context"] for row in pair_rows}
if pair_contexts != TEST_CONTEXTS | DEVELOPMENT_CONTEXTS:
    raise ValueError(f"Unexpected E0 contexts: {sorted(pair_contexts)}")
if TEST_CONTEXTS & DEVELOPMENT_CONTEXTS:
    raise ValueError("Test and development contexts overlap.")

print(f"RGB pairs: {len(pair_rows):,}")
print(f"Development pairs: {sum(row['partition'] == 'development' for row in pair_rows):,}")
print(f"Test pairs: {sum(row['partition'] == 'test' for row in pair_rows):,}")
print("Frozen recording-level assignments verified; no new split created.")

In [ ]:
def materialize_pairs(rows: list[dict[str, str]], workspace: Path) -> Path:
    if workspace.exists():
        shutil.rmtree(workspace)
    for partition in ("development", "test"):
        (workspace / "images" / partition).mkdir(parents=True, exist_ok=True)
        (workspace / "labels" / partition).mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        member_names = {info.filename.replace("\\", "/") for info in archive.infolist()}
        for row in sorted(rows, key=lambda item: (item["partition"], item["image_member_path"])):
            image_member = row["image_member_path"]
            annotation_member = row["annotation_member_path"]
            if image_member not in member_names or annotation_member not in member_names:
                raise FileNotFoundError(f"Manifest member missing from ZIP: {image_member} / {annotation_member}")
            image_target = workspace / "images" / row["partition"] / Path(image_member).name
            label_target = workspace / "labels" / row["partition"] / f"{Path(image_member).stem}.txt"
            image_target.write_bytes(archive.read(image_member))
            label_target.write_bytes(archive.read(annotation_member))

    with (workspace / "pair_manifest.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=PAIR_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    return workspace

E0_WORKSPACE = materialize_pairs(pair_rows, E0_ROOT)
print(f"Materialized validated pairs into {E0_WORKSPACE}")
print("Only manifest-referenced members were extracted; the original ZIP was not modified.")

## Development grouped cross-validation and final test

The four CV folds below are context-level only. Final test contexts are never used in CV or training. After CV, the final YOLOv8s model is trained on all development pairs with validation disabled, then evaluated exactly once on the frozen test pairs.

In [ ]:
def write_yolo_data_yaml(path: Path, train_list: Path, val_list: Path | None) -> None:
    lines = [f"train: {train_list.as_posix()}"]
    if val_list is not None:
        lines.append(f"val: {val_list.as_posix()}")
    lines.extend(["nc: 1", "names: ['person']"])
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_image_list(path: Path, rows: list[dict[str, str]]) -> None:
    paths = [str((E0_WORKSPACE / "images" / row["partition"] / Path(row["image_member_path"]).name).resolve()) for row in rows]
    path.write_text("\n".join(paths) + "\n", encoding="utf-8")


def rows_for_contexts(contexts: set[str]) -> list[dict[str, str]]:
    return [row for row in pair_rows if row["collection_context"] in contexts]


In [ ]:
try:
    from ultralytics import YOLO
except ImportError as error:
    raise ImportError("Install ultralytics in Colab before running E0 training: pip install ultralytics") from error

CV_RESULTS = []
for fold_number, validation_context in enumerate(sorted(DEVELOPMENT_CONTEXTS), start=1):
    fold_root = E0_ROOT / "cv" / f"fold_{fold_number}"
    fold_root.mkdir(parents=True, exist_ok=True)
    validation_rows = rows_for_contexts({validation_context})
    training_rows = rows_for_contexts(DEVELOPMENT_CONTEXTS - {validation_context})
    train_list = fold_root / "train_images.txt"
    val_list = fold_root / "val_images.txt"
    yaml_path = fold_root / "data.yaml"
    write_image_list(train_list, training_rows)
    write_image_list(val_list, validation_rows)
    write_yolo_data_yaml(yaml_path, train_list, val_list)
    assert not ({row["collection_context"] for row in training_rows} & {validation_context})
    assert not ({row["collection_context"] for row in validation_rows} & TEST_CONTEXTS)

    if RUN_TRAINING:
        model = YOLO("yolov8s.pt")
        result = model.train(
            data=str(yaml_path),
            project=str(E0_ROOT / "runs"),
            name=f"cv_fold_{fold_number}",
            epochs=100,
            imgsz=640,
            seed=20260905,
            patience=20,
            val=True,
        )
        CV_RESULTS.append({"fold": fold_number, "validation_context": validation_context, "result": result})

print(f"Prepared {len(DEVELOPMENT_CONTEXTS)} grouped development CV folds.")
print("Final test contexts were excluded from every fold.")

In [ ]:
final_root = E0_ROOT / "final"
final_root.mkdir(parents=True, exist_ok=True)
development_rows = rows_for_contexts(DEVELOPMENT_CONTEXTS)
test_rows = rows_for_contexts(TEST_CONTEXTS)
final_train_list = final_root / "development_images.txt"
test_list = final_root / "test_images.txt"
final_yaml = final_root / "development_data.yaml"
test_yaml = final_root / "test_data.yaml"
write_image_list(final_train_list, development_rows)
write_image_list(test_list, test_rows)
write_yolo_data_yaml(final_yaml, final_train_list, None)
test_yaml.write_text(
    f"path: {E0_WORKSPACE.as_posix()}\n"
    f"val: {test_list.as_posix()}\n"
    "names:\n  0: person\n",
    encoding="utf-8",
)

if RUN_TRAINING:
    final_model = YOLO("yolov8s.pt")
    final_model.train(
        data=str(final_yaml),
        project=str(E0_ROOT / "runs"),
        name="final_development",
        epochs=100,
        imgsz=640,
        seed=20260905,
        patience=0,
        val=False,
    )
    IMAGE_SIZE = 640
    BATCH_SIZE = 8
    DEVICE = 0 if __import__("torch").cuda.is_available() else "cpu"
    WORKERS = 2
    test_metrics = final_model.val(
        data=str(test_yaml),
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        split="val",
    )
    print("Final model evaluated once on the frozen test pairs.")
else:
    print("RUN_TRAINING=False: prepared final train/test references without training.")

In [ ]:
print("E0 RGB-only baseline configuration")
print(f"Validated RGB pairs: {len(pair_rows):,}")
print(f"Development pairs: {len(development_rows):,}")
print(f"Final test pairs: {len(test_rows):,}")
print(f"Development contexts: {sorted(DEVELOPMENT_CONTEXTS)}")
print(f"Final test contexts: {sorted(TEST_CONTEXTS)}")
print("No chronological split or alternate manifest was used.")
print("Final test data was excluded from CV and training, then evaluated once.")